# 单元二：自动求导机制与计算图 (Autograd & Graphs)

---

## 单元 2.1：动态计算图 —— 梯度的“生命周期”

在“鱼书”笔记第 5 章中，你推导过复杂的矩阵微分。在 PyTorch 中，这一切被抽象为一棵**有向无环图 (DAG)**。

### 1. 核心定义：梯度的追踪器

*   **`requires_grad`**：这是一个布尔标记。如果设为 `True`，PyTorch 会开始追踪该张量上的所有运算，并构建计算图。
*   **`grad_fn`**：这是张量的“出生证明”。它记录了该张量是通过什么运算产生的（如 `AddBackward`, `MulBackward`）。
*   **`backward()`**：触发器。调用它时，PyTorch 会根据**链式法则**，从输出节点反向遍历计算图，计算所有叶子节点的梯度。
*   **`.grad`**：累加器。计算出的偏导数 $\frac{\partial out}{\partial x}$ 会存储在这里。

### 2. 实验：追踪一个复合函数

请在 Jupyter 中运行以下代码，观察偏导数的自动计算过程：

In [1]:
import torch

# 1. 创建输入张量，并开启梯度追踪
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# 2. 构建计算过程 (计算图在这一步动态生成)
# y = w * x + b
y = w * x + b

# 3. 查看 grad_fn
print(f"y 的产生方式 (grad_fn): {y.grad_fn}") 

# 4. 执行反向传播
y.backward()

# 5. 查看偏导数
# dy/dx = w = 3.0
# dy/dw = x = 2.0
# dy/db = 1.0
print(f"dy/dx: {x.grad}")
print(f"dy/dw: {w.grad}")
print(f"dy/db: {b.grad}")

y 的产生方式 (grad_fn): <AddBackward0 object at 0x70e09c418c10>
dy/dx: tensor([3.])
dy/dw: tensor([2.])
dy/db: tensor([1.])


在模块二中，单元 2.1 是整个深度学习框架的“灵魂”所在。你之前看“鱼书”笔记时，第 5 章手推的反向传播逻辑，在 PyTorch 里全部被集成到了这个 **Autograd（自动求导系统）** 中。

为了让你彻底理解 2.1 的实验内容，我们从**数学本质**、**程序逻辑**和**竞赛应用**三个维度进行深度拆解。

---

#### 1. 深度定义解释：Autograd 的“三驾马车”

在实验代码中，这三个属性共同定义了一个张量在计算图中的角色：

*   **`requires_grad=True`（身份标记）**：
    *   **本质**：告诉 PyTorch 的计算引擎：“这个张量是可变的，请为它开辟一块内存空间来记录它的导数（`.grad`），并追踪所有涉及它的运算。”
    *   **竞赛对标**：在光子 AI 模型中，你的权重矩阵 $W$ 必须开启这个标记，否则模型无法根据误差进行更新（学习）。

*   **`grad_fn`（运算溯源/导数函数）**：
    *   **本质**：它是张量的“出生证明”。当你执行 `y = w * x` 时，产生的 `y` 会自带一个 `MulBackward` 属性。
    *   **逻辑**：它存储了如何计算该操作导数的**数学公式**。在反向传播时，PyTorch 并不需要重新推导公式，它只需要查阅 `grad_fn` 里的“导数说明书”，代入数值即可。

*   **`backward()`（连锁反应开关）**：
    *   **本质**：它是**链式法则（Chain Rule）**的程序化触发器。
    *   **动作**：当你对最终结果（通常是 Loss）调用 `.backward()` 时，它会从最后一片叶子开始，顺着 `grad_fn` 像推倒多米诺骨牌一样，把导数一层层传回到最开始的输入张量。

---

#### 2. 实验步骤的逻辑解剖

我们来看那个公式：$y = w \cdot x + b$

##### **第一阶段：前向传播（Forward Pass）与 动态建图**
当你执行代码 `y = w * x + b` 时，PyTorch 内部发生了两件事：
1.  **数值计算**：算出 $3.0 \times 2.0 + 1.0 = 7.0$。
2.  **动态建图**：在内存中瞬间搭建了一棵树（图）：
    *   底端是叶子节点：`w`($3.0$), `x`($2.0$), `b`($1.0$)。
    *   中间节点：`mul` 运算（产生了中间变量 `temp = w * x`）。
    *   顶端节点：`add` 运算（产生了最终变量 `y`）。
    *   **注意**：这棵图是“动态”的，每次运行 `y = ...` 都会重新构建，这比 TensorFlow 的静态图灵活得多。

##### **第二阶段：反向传播（Backward Pass）**
当你执行 `y.backward()` 时，计算引擎开始工作：
1.  它看 `y` 的 `grad_fn`，发现是 `AddBackward`。
2.  根据加法求导法则：$\frac{\partial y}{\partial (w \cdot x)} = 1$ 和 $\frac{\partial y}{\partial b} = 1$。
3.  接着它看中间变量 `temp = w \cdot x` 的 `grad_fn`，发现是 `MulBackward`。
4.  根据乘法求导法则（链式法则）：
    *   $\frac{\partial y}{\partial w} = \frac{\partial y}{\partial temp} \cdot \frac{\partial temp}{\partial w} = 1 \cdot x = 2.0$
    *   $\frac{\partial y}{\partial x} = \frac{\partial y}{\partial temp} \cdot \frac{\partial temp}{\partial x} = 1 \cdot w = 3.0$
5.  **结果存储**：最终算出的 $2.0, 3.0, 1.0$ 分别被塞进了 `w.grad`, `x.grad`, `b.grad` 中。



---

## 单元 2.2：梯度累加与清除

这是 PyTorch 初学者最容易犯错的地方。

*   **累加特性**：在默认情况下，PyTorch **不会**在每次 `backward()` 后清空 `.grad`。如果你多次运行反向传播，梯度会不断叠加。
*   **原因**：这在处理超大规模 Batch（分批梯度累加）时非常有用，但在常规训练中，我们必须在每次迭代前清空梯度。

#### **实验：观察累加现象**

In [2]:
# 再次运行一次反向传播（假设 y 是重新计算的）
y = w * x + b
y.backward()
print(f"第二次运行后的 dy/dx: {x.grad}") # 你会发现结果变成了 6.0 (3.0 + 3.0)

# 清零操作 (在模块五中，优化器 zero_grad() 会自动做这件事)
x.grad.zero_()
print(f"清零后的 dy/dx: {x.grad}")

第二次运行后的 dy/dx: tensor([6.])
清零后的 dy/dx: tensor([0.])



---

## 单元 2.3：上下文管理 —— `no_grad()` 与推理模式

在光子计算的**实时推理**场景中，我们只需要结果，不需要计算梯度。此时，构建计算图不仅浪费内存，还会拖慢 CPU/GPU 速度。

### 1. 核心操作：`torch.no_grad()`

这是一个上下文管理器，它会强制关闭内部所有运算的梯度追踪。

In [3]:
x = torch.tensor([2.0], requires_grad=True)

with torch.no_grad():
    y = x * 2
    print(f"推理模式下 y 是否追踪梯度: {y.requires_grad}")
    print(f"推理模式下 y 的 grad_fn: {y.grad_fn}") # 输出 None

推理模式下 y 是否追踪梯度: False
推理模式下 y 的 grad_fn: None



### 2. `torch.inference_mode()` —— 极速推理模式 (更专业的选择)

这是 PyTorch 1.9+ 引入的新特性。它是 `no_grad` 的**升级版**。

*   **区别**：`no_grad` 只是停止记录梯度；而 `inference_mode` 会在底层进行更多优化（例如跳过版本号追踪等运行时开销）。
*   **性能**：比 `no_grad` **更快**。
*   **竞赛应用**：在你的 `evaluate_mnist.py` 或未来的光子芯片实时推理脚本中，你应该优先使用这个。

In [4]:
with torch.inference_mode():
    y = x * 2
    # 这里运行速度最快，显存占用最低

### 3. `torch.set_grad_enabled(mode)` —— 动态开关

这是一个灵活的全局开关，接收一个布尔值。

*   **场景**：当你写一个通用的函数，既要负责训练又要负责验证时非常有用。
*   **代码示例**：

In [ ]:
is_train = False # 动态决定
with torch.set_grad_enabled(is_train):
    pass
#   output = model(input_data)

### 4. `tensor.detach()` —— 手动断开连接 (剪枝操作)

虽然它不是上下文管理器，但逻辑上属于“梯度控制”。它的作用是创建一个**数值完全一样**但 **`requires_grad=False`** 的新张量，且这个新张量脱离了原有的计算图。

*   **物理意义**：在计算图中手动剪断一根树枝。
*   **竞赛场景（光子硬件反馈循环）**：
    如果你在训练过程中需要把某个中间结果保存下来做统计，但不希望这个统计过程被求导，你就需要用 `detach()`。

In [ ]:
# 假设 output 是光子层的输出
# 我们想把结果转成 NumPy 绘图，必须先 detach，因为有梯度的张量不能直接转 numpy
# output_fixed = output.detach().cpu().numpy()


---

## 单元 2.4：叶子张量与 In-place 操作

### 1. 叶子张量（Leaf Tensor）的严格定义
在计算图中，**叶子张量**是所有计算的“源头”。

*   **判定标准**：
    1.  所有 `requires_grad=False` 的张量。
    2.  用户显式创建的 `requires_grad=True` 的张量（如模型权重 `W`）。
    3.  通过 `.detach()` 产生的张量。
*   **属性查看**：可以使用 `t.is_leaf` 查看。
*   **为什么重要？**：为了节省内存，PyTorch 的 `backward()` 在默认情况下**只为叶子张量保留 `.grad`**。如果你尝试访问中间变量（非叶子张量）的梯度，会得到 `None`。

### 2. In-place 操作的“版本计数器”机制
你可能会好奇：为什么 PyTorch 能精准捕捉到你修改了数据？

*   **Version Counter**：每个张量内部都有一个版本计数器。
*   **机制**：每当张量被 In-place 修改，计数器就 `+1`。
*   **冲突检测**：计算图在 `forward` 阶段会记录当前张量的版本号。如果在 `backward` 时发现计数器变了，说明计算导数所需的“原始现场”被破坏了，于是立刻抛出 `RuntimeError`。

### 3. 何时允许使用 In-place 操作？
虽然 In-place 操作有风险，但它能**节省大量显存**（不需要创建新对象）。在以下情况是安全的：

*   **在 `requires_grad=False` 的张量上**：既然不需要求导，随便改。
*   **在 `torch.no_grad()` 上下文中**：临时关闭了梯度追踪，修改是安全的。
*   **特定的激活函数**：例如 `nn.ReLU(inplace=True)`。
    *   *深度解释*：ReLU 的导数逻辑非常特殊——它只取决于输出是否大于 0。即使你覆盖了输入值，只要输出还在，导数就能算出来。

### 4. 竞赛实战：手动更新权重的标准姿势
在光子计算竞赛中，如果你需要手写一个特定的优化器算法（比如模拟退火或自定义 SGD），你会遇到这个矛盾：**你想修改权重（Leaf），但 Git 不允许你 In-place 修改。**

**❌ 错误的写法（导致权重不再是叶子）：**
```python
W = W - lr * W.grad  # W 变成了运算结果，is_leaf 会变成 False，梯度链断裂
```

**✅ 专业的写法（保持叶子身份）：**
```python
with torch.no_grad():
    W.sub_(lr * W.grad)  # 使用下划线操作，并在 no_grad 中执行
```

---

### 🚀 模块二 练习任务

请在 Jupyter Notebook 中完成此练习。

#### 任务 A：计算图构建与梯度校验（对应 2.1 & 2.2）
1.  **定义张量**：创建标量 $a = 2.0$ 和 $b = 3.0$，并开启梯度追踪。创建标量 $c = 4.0$，不开启梯度追踪。
2.  **复合运算**：计算 $x = a + b$，然后计算 $L = x \times c$。
3.  **反向传播**：对 $L$ 调用 `.backward()`。
4.  **结果核对**：
    *   输出 `a.grad` 和 `b.grad`。手动计算 $\frac{\partial L}{\partial a} = c = 4.0$，看程序结果是否一致。
    *   输出 `x.grad`。你会发现它是 `None`，请说明为什么？（提示：$x$ 是**叶子张量**还是**非叶子张量**？）
5.  **梯度累加验证**：不执行清零操作，再次执行 `L = (a + b) * c` 和 `L.backward()`。
    *   再次输出 `a.grad`，观察其值变成了多少？说明了 PyTorch 梯度的什么特性？

In [5]:
# A
import torch

a = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([3.0], requires_grad=True)
c = torch.tensor([4.0], requires_grad=False)

x = a + b
L = x * c
L.backward()

print(f"dy/da: {a.grad}") # 4.0
print(f"dy/db: {b.grad}") # 4.0
print(f"dy/dx: {x.grad}") # None，因为 x 是中间变量，默认不保留梯度, x是非叶子张量

L = (a + b) * c
L.backward()
print(f"dy/da: {a.grad}") # 8.0, 说明Pytorch梯度具有累加特性

dy/da: tensor([4.])
dy/db: tensor([4.])
dy/dx: None
dy/da: tensor([8.])


/tmp/ipykernel_177889/2780364440.py:14: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  print(f"dy/dx: {x.grad}") # None，因为 x 是中间变量，默认不保留梯度, x是非叶子张量


#### 任务 B：上下文管理与隔离测试（对应 2.3）
1.  **推理模式测试**：使用 `with torch.inference_mode():` 包裹运算 $y = a \times b$。
    *   打印 `y.requires_grad` 和 `y.grad_fn`，确认在该模式下是否还存在计算图。
2.  **张量剪枝**：令 $d = a.detach()$。
    *   修改 $d$ 的值（例如 `d.zero_()`），观察 $a$ 的值是否发生了变化？
    *   检查 $d.requires\_grad$，确认它是否脱离了计算图。

In [4]:
# B
a = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([3.0], requires_grad=True)
with torch.inference_mode():
    y = a + b
    print(f"推理模式下 y 是否追踪梯度: {y.requires_grad}")
    print(f"推理模式下 y 的 grad_fn: {y.grad_fn}")

d = a.detach()
d.zero_() # 这不会影响 a 的梯度，因为 d 是 a 的一个副本，且不追踪梯度
print(a)
print(f"{d.requires_grad=}")

推理模式下 y 是否追踪梯度: False
推理模式下 y 的 grad_fn: None
tensor([0.], requires_grad=True)
d.requires_grad=False


#### 任务 C：内存安全性与版本计数器（对应 2.4）
1.  **报错实验**：
    *   创建 $e = torch.tensor([1.0], requires_grad=True)$。
    *   计算 $f = e ** 2$。
    *   在执行 `f.backward()` **之前**，对 $e$ 执行一次 In-place 修改：`e.add_(1.0)`。
    *   运行代码并观察报错信息。
2.  **思考题**：为什么对于开启了 `requires_grad` 的叶子张量，PyTorch 要通过“版本计数器（Version Counter）”禁止这种 In-place 操作？

In [ ]:
# C 1
import torch

e = torch.tensor([1.0], requires_grad=True)
f = e ** 2
e.add_(1.0) # RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.
f.backward()

# C 2
# 因为修改这种叶子张量会破坏计算图，导致无法正确计算梯度，所以 PyTorch 不允许对 requires_grad=True 的叶子张量进行原地修改。

RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.

你的练习报告非常出色，特别是对 **梯度累加特性** 和 **In-place 报错机制** 的捕捉非常准确。

针对你在 **任务 B** 中提出的疑问，这是一个非常经典的 **“内存共享 vs. 数据拷贝”** 的误区。我为你深度复盘一下：

### 1. 深度复盘：为什么 `a` 变成了 0？

你问：“a 的梯度已经在 A 中变成了 8.0，为什么 B 中会得到 0 的结果，不应该不会被 d 影响吗？”

这里有两个关键概念需要剥离：
1.  **梯度（`.grad`）**：这是存储导数的地方。在 A 结束后，`a.grad` 确实是 `8.0`。
2.  **数值（`.data` 或 `a` 本身）**：这是张量存储的原始数据。在 A 运行前，`a` 的值是 `2.0`。

**核心原因**：`detach()` 产生的张量与原张量 **共享底层的内存空间（Storage）**。
*   `d = a.detach()`：PyTorch 并没有在内存里复制一份新的数据，它只是创建了一个新的“逻辑外壳” `d`，并把 `d` 的 `requires_grad` 设为 `False`。
*   `d.zero_()`：这是一个 **In-place（原地）** 操作。既然 `d` 和 `a` 共享同一块内存，当你把 `d` 抹成 0 时，`a` 对应地址上的值也被抹成了 0。

**结论**：`a.grad` 依然是 `8.0`（如果你打印 `print(a.grad)` 就能看到），但 `a` 的 **数值** 已经变成了 `0`。